# MambaDSR — Jasper WER on UA-Speech B2

Replicates UNIT-DSR's objective evaluation: NVIDIA Jasper (`stt_en_jasper10x5dr`), greedy CTC decoding,
WER on block **B2** (held out from training), per speaker, original vs reconstructed speech.

This notebook only **defines** `run_jasper(checkpoint, ...)`; nothing runs on import. It expects `main.py`
to be loaded (`SpeechReconstructionPipeline`, model constants, `UA_PATH`, `UA_RAW_PATH`) and NeMo installed
with torch pinned:

```
!pip freeze | grep -E "^(torch|torchaudio|torchvision|torchcodec)==" > /tmp/torch_pin.txt
!pip install -q -c /tmp/torch_pin.txt "nemo_toolkit[asr]" jiwer xlrd
```

Use a checkpoint from the **B1+B3** run; older ones saw B2 during training and give inflated WER.

In [ ]:
WORDLIST    = "uaspeech/speaker_wordlist.csv"
SPEAKERS    = ["M05", "F04", "M07", "F02"]    # UNIT-DSR's evaluation speakers
MIC         = "M5"                            # one channel, so each word counts once (None = all)
OUT_DIR     = "/content/b2_eval"              # generated wavs (local disk)
RESULTS_DIR = "results"                       # CSVs; point at Drive to keep them

## Official severity

Taken from the corpus' own speaker table (`Speaker` sheet of `speaker_wordlist.xls`, intelligibility in %).
Use this for the paper rather than the hand-written `SEVERITY` in `DataLoaders.py`, which disagrees for several speakers.

In [ ]:
OFFICIAL_SEVERITY = {
    "M01": "very low", "M04": "very low", "M12": "very low", "F03": "very low",
    "M07": "low", "F02": "low", "M16": "low",
    "M05": "mid", "F04": "mid", "M11": "mid",
    "M08": "high", "M09": "high", "M10": "high", "M14": "high", "F05": "high",
}
INTELLIGIBILITY = {   # % words correctly transcribed by listeners
    "M01": 15, "M04": 2, "M12": 7.4, "F03": 6, "M07": 28, "F02": 29, "M16": 43,
    "M05": 58, "F04": 62, "M11": 62, "M08": 93, "M09": 86, "M10": 93, "M14": 90.4, "F05": 95,
}


## Evaluation code

In [ ]:
import re
from pathlib import Path

import jiwer
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio.functional as AF
from tqdm.auto import tqdm

EVAL_SR = 16000                                # Jasper expects 16 kHz mono
UNIT_DSR_SPEAKERS = ("M05", "F04", "M07", "F02")
SYSTEMS = ("original", "vocoded", "reconstructed")


# Helpers
def _show(df):
    try:
        from IPython.display import display
        display(df)
    except ImportError:
        print(df.to_string(index=False))


def _parse_stem(stem):
    """'F02_B2_CW18_M5' -> ('F02', 'B2', 'CW18', 'M5')."""
    parts = stem.split("_")
    return parts[0], parts[1], "_".join(parts[2:-1]), parts[-1]


def _save_16k(waveform, sr, path):
    """Write any (…, samples) tensor as 16 kHz mono float wav."""
    w = torch.as_tensor(waveform).reshape(1, -1).float().cpu()
    w = AF.resample(w, sr, EVAL_SR).clamp(-1.0, 1.0)
    sf.write(str(path), w.squeeze(0).numpy(), EVAL_SR)


def normalize(text):
    """Lower-case, keep letters and apostrophes, collapse spaces (refs and hyps alike)."""
    text = str(text).lower().replace("-", " ")
    text = re.sub(r"[^a-z' ]", " ", text)
    return " ".join(text.split())


# Word labels, read from the corpus word list
def load_word_labels(wordlist_path):
    """
    Map UA-Speech file codes to words using doc/speaker_wordlist.xls.
    Digits, letters, commands and common words share a code across blocks ("CW18");
    uncommon words differ per block, so they are keyed with the block ("B2_UW1").
    """
    if str(wordlist_path).endswith(".csv"):
        df = pd.read_csv(wordlist_path)
    else:   # the official .xls has a "Speaker" sheet first; the labels are on "Word_filename"
        df = pd.read_excel(wordlist_path, sheet_name="Word_filename")
    cols = {c.lower().strip(): c for c in df.columns}
    try:
        word_col = next(cols[c] for c in cols if "word" in c)
        code_col = next(cols[c] for c in cols if "file" in c or "name" in c)
    except StopIteration:
        raise ValueError(f"Unexpected word-list columns: {list(df.columns)}")

    labels = {
        str(code).strip().upper(): str(word).strip()
        for code, word in zip(df[code_col], df[word_col])
        if not (pd.isna(code) or pd.isna(word))
    }
    print(f"Loaded {len(labels)} word labels")
    return labels


def label_for(labels, block, word_code):
    for key in (f"{block}_{word_code}".upper(), word_code.upper()):
        if key in labels:
            return labels[key]
    return None


# Step 1 — reconstruct B2 and write 16 kHz wavs
@torch.no_grad()
def reconstruct_b2(
    pipeline,
    speaker_embs,
    mel_dir,
    raw_dir,
    out_dir="b2_eval",
    speakers=UNIT_DSR_SPEAKERS,
    mic="M5",
    vocode_original=True,
    vocoder_sr=22050,
    vocoder_scale=0.5,
):
    """
    For every B2 utterance of `speakers` on microphone `mic`, write:
      original/       raw dysarthric recording            (the ASR baseline)
      vocoded/        ground-truth mel -> HiFi-GAN        (vocoder ceiling, optional)
      reconstructed/  mel -> your model -> HiFi-GAN
    One mic only, so each word is counted once. mic=None uses every channel.
    vocoder_scale: mels are log POWER (power=2); the LJSpeech HiFi-GAN expects log magnitude,
    so both vocoded and reconstructed mels are multiplied by 0.5 first (same for every system).
    """
    device = next(pipeline.parameters()).device
    pipeline.eval()                     # also stops VQ EMA updates / dead-code resets

    out = Path(out_dir)
    systems = ["original", "reconstructed"] + (["vocoded"] if vocode_original else [])
    for s in systems:
        (out / s).mkdir(parents=True, exist_ok=True)

    for sid in speakers:
        if sid not in speaker_embs:
            print(f"  {sid}: no speaker embedding, skipped")
            continue
        spk_emb = speaker_embs[sid].to(device).reshape(1, -1)

        pattern = f"{sid}_B2_*_{mic}.pt" if mic else f"{sid}_B2_*.pt"
        mel_files = sorted(Path(mel_dir).rglob(pattern))

        spk_raw = Path(raw_dir) / sid
        root = spk_raw if spk_raw.exists() else Path(raw_dir)
        raw_index = {p.stem: p for p in root.rglob(f"{sid}_B2_*.wav")}

        missing_raw = 0
        for mel_path in tqdm(mel_files, desc=sid):
            stem = mel_path.stem
            raw_path = raw_index.get(stem)
            if raw_path is None:
                missing_raw += 1
                continue

            # Original recording, untouched apart from mono + resampling
            audio, sr = sf.read(str(raw_path), dtype="float32", always_2d=True)
            _save_16k(torch.from_numpy(audio.mean(axis=1)), sr, out / "original" / f"{stem}.wav")

            # Reconstruction — signature: (mel, padding_mask, speaker_ids, speaker_emb, target, run_vocoder)
            mel = torch.load(mel_path, weights_only=False)["mel"].unsqueeze(0).to(device)
            T = mel.shape[-1]
            result = pipeline(
                mel,
                torch.zeros(1, T, dtype=torch.bool, device=device),   # no padding
                torch.zeros(1, dtype=torch.long, device=device),      # adversary label, unused here
                spk_emb,                                              # recognizer ckpts use CF02's voice
                None,
                False,
            )
            _save_16k(pipeline.mel_to_waveform(vocoder_scale * result["predicted"]), vocoder_sr,
                      out / "reconstructed" / f"{stem}.wav")

            if vocode_original:
                _save_16k(pipeline.mel_to_waveform(vocoder_scale * mel), vocoder_sr, out / "vocoded" / f"{stem}.wav")

        print(f"  {sid}: {len(mel_files) - missing_raw} utterances"
              + (f" ({missing_raw} without a raw wav, skipped)" if missing_raw else ""))

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out


# Step 2 — Jasper transcription and WER
def load_jasper(device=None):
    import nemo.collections.asr as nemo_asr
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = nemo_asr.models.EncDecCTCModel.from_pretrained(
        "stt_en_jasper10x5dr", map_location=torch.device(device)
    )
    return model.eval()


def transcribe(model, paths, batch_size=16):
    """Greedy CTC decoding (EncDecCTCModel's default)."""
    outputs = model.transcribe(paths, batch_size=batch_size)
    if isinstance(outputs, tuple):        # some NeMo versions return (best, all)
        outputs = outputs[0]
    return [getattr(o, "text", o) for o in outputs]


def wer_table(df, by, systems):
    """Corpus WER per group plus a pooled 'ALL' row. Insertions count, so WER can exceed 100%."""
    rows = []
    for key, g in list(df.groupby(by)) + [("ALL", df)]:
        row = {by: key, "n_words": len(g)}
        for s in systems:
            row[f"WER_{s}"] = round(100 * jiwer.wer(g["reference"].tolist(), g[f"hyp_{s}"].tolist()), 1)
        # single-word references: exact match and CER are the readable metrics (WER rewards empty output)
        for s in systems:
            row[f"exact_{s}"] = round(100 * (g[f"hyp_{s}"] == g["reference"]).mean(), 1)
            row[f"CER_{s}"] = round(100 * jiwer.cer(g["reference"].tolist(), g[f"hyp_{s}"].tolist()), 1)
        o, r = row["WER_original"], row["WER_reconstructed"]
        row["rel_reduction_%"] = round(100 * (o - r) / o, 1) if o else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def score_b2(out_dir, wordlist_path, severity_of=None, batch_size=16, device=None):
    out = Path(out_dir)
    labels = load_word_labels(wordlist_path)

    rows = []
    for p in sorted((out / "original").glob("*.wav")):
        speaker, block, word_code, mic = _parse_stem(p.stem)
        rows.append({
            "stem": p.stem,
            "speaker": speaker,
            "severity": (severity_of or {}).get(speaker),
            "word_code": word_code,
            "mic": mic,
            "reference": label_for(labels, block, word_code),
        })
    df = pd.DataFrame(rows)

    missing = df["reference"].isna()
    if missing.any():
        print(f"WARNING: no label for {missing.sum()} files "
              f"(e.g. {df.loc[missing, 'word_code'].unique()[:5].tolist()}), dropped")
        df = df[~missing].reset_index(drop=True)
    df["reference"] = df["reference"].map(normalize)

    systems = [s for s in SYSTEMS if (out / s).is_dir()]
    model = load_jasper(device)
    for s in systems:
        paths = [str(out / s / f"{stem}.wav") for stem in df["stem"]]
        df[f"hyp_{s}"] = [normalize(t) for t in transcribe(model, paths, batch_size)]

    print("\nJasper WER (%) per speaker — UA-Speech B2")
    per_speaker = wer_table(df, "speaker", systems)
    _show(per_speaker)

    if df["severity"].notna().any():
        print("\nJasper WER (%) per severity")
        _show(wer_table(df.dropna(subset=["severity"]), "severity", systems))

    spk_only = per_speaker[per_speaker["speaker"] != "ALL"]
    print(f"\nMean of per-speaker relative reductions: {spk_only['rel_reduction_%'].mean():.1f}%")
    return df, per_speaker


# One call does both
def evaluate_b2(
    pipeline,
    speaker_embs,
    mel_dir,
    raw_dir,
    wordlist_path,
    out_dir="b2_eval",
    speakers=UNIT_DSR_SPEAKERS,
    mic="M5",
    severity_of=None,
    vocode_original=True,
    batch_size=16,
):
    reconstruct_b2(pipeline, speaker_embs, mel_dir, raw_dir, out_dir,
                   speakers=speakers, mic=mic, vocode_original=vocode_original)
    return score_b2(out_dir, wordlist_path, severity_of=severity_of, batch_size=batch_size)

## `run_jasper`

In [ ]:
import os
from pathlib import Path
import torch


def load_pipeline(checkpoint, device="cuda:0"):
    """Build the pipeline exactly as train() does and load `checkpoint`. Needs main.py loaded."""
    ckpt = torch.load(checkpoint, map_location=device, weights_only=False)
    state = ckpt["model_state_dict"]
    # number of speakers = output size of the adversary's last layer (state_dict keeps module order)
    adv_weights = [k for k in state if k.startswith("speaker_adversary") and k.endswith("weight")]

    pipeline = SpeechReconstructionPipeline(
        n_mels=N_MELS, d_model=D_MODEL, n_layers=N_LAYERS,
        **codebook_arch(state), spk_dim=SPK_DIM,          # sizes from the checkpoint
        num_speakers=state[adv_weights[-1]].shape[0], speaker_adv_weight=0.1, device=device,
    ).to(device)
    missing, unexpected = pipeline.load_state_dict(state, strict=False)
    # code_heads / reference_emb only exist in recognizer checkpoints
    missing = [k for k in missing if not k.startswith(("code_heads", "reference_emb"))]
    if missing or unexpected:   # e.g. skip_proj missing = main.py is older than the checkpoint
        print("WARNING  missing:", missing, "\n         unexpected:", unexpected)
    # Old checkpoints don't store the flags: they were trained with the skip path, without code heads
    pipeline.use_skip = ckpt.get("use_skip", True)
    pipeline.predict_codes = ckpt.get("predict_codes", False)
    pipeline.eval()

    print(f"Loaded {checkpoint} (skip {'on' if pipeline.use_skip else 'off'}, "
          f"{'predicted codes' if pipeline.predict_codes else 'VQ codes'})"
          f" (step {ckpt.get('step')}, mode {ckpt.get('mode')}, "
          f"best val {ckpt.get('best_metric')})")
    return pipeline, ckpt["speaker_embeddings"]


def run_jasper(checkpoint, speakers=SPEAKERS, mic=MIC, device="cuda:0", save=True,
               decode="argmax", smooth_width=2, vocode_original=True):
    """Jasper WER for `checkpoint`. speakers="all" uses every dysarthric speaker with an embedding.

    decode / smooth_width: how a recognizer checkpoint turns predicted codes into decoder input
    ("argmax", "smooth", "soft", "smooth_soft"); ignored for other checkpoints.
    vocode_original=False skips the vocoded system (it doesn't depend on the checkpoint).
    """
    pipeline, speaker_embs = load_pipeline(checkpoint, device)
    pipeline.decode, pipeline.smooth_width = decode, smooth_width
    if speakers == "all":
        speakers = sorted(s for s in speaker_embs if s in OFFICIAL_SEVERITY)

    name = Path(checkpoint).stem
    if pipeline.predict_codes and decode != "argmax":
        name += f"_{decode}" + (f"{smooth_width}" if "smooth" in decode else "")
    label = "unitdsr" if list(speakers) == list(SPEAKERS) else f"{len(speakers)}spk"
    df, per_speaker = evaluate_b2(
        pipeline, speaker_embs,
        mel_dir=UA_PATH, raw_dir=UA_RAW_PATH, wordlist_path=WORDLIST,
        out_dir=f"{OUT_DIR}/{name}_{label}", speakers=speakers, mic=mic,
        severity_of=OFFICIAL_SEVERITY, vocode_original=vocode_original,
    )

    if save:
        os.makedirs(RESULTS_DIR, exist_ok=True)
        df.to_csv(f"{RESULTS_DIR}/jasper_{name}_{label}_utterances.csv", index=False)
        per_speaker.to_csv(f"{RESULTS_DIR}/jasper_{name}_{label}_per_speaker.csv", index=False)

    del pipeline
    torch.cuda.empty_cache()
    return df, per_speaker

In [ ]:
def spot_check(df, n=15):
    """Words the reconstruction fixed, and words it broke."""
    fixed  = df[(df.hyp_original != df.reference) & (df.hyp_reconstructed == df.reference)]
    broken = df[(df.hyp_original == df.reference) & (df.hyp_reconstructed != df.reference)]
    print(f"fixed {len(fixed)}, broken {len(broken)}")
    cols = ["speaker", "reference", "hyp_original", "hyp_reconstructed"]
    return fixed[cols].head(n), broken[cols].head(n)

## codes2mel gate

Go/no-go for the code-prediction design. A healthy speaker's B2 words go through the frozen encoder + VQ,
then the codes-only decoder (skip zeroed), then HiFi-GAN. `direct` vocodes the same mel without the model:
the ceiling. Pass if `codes` comes close to `direct`. Both use the 0.5× power-2 → magnitude approximation.

In [ ]:
@torch.no_grad()
def codes2mel_gate(checkpoint, reference="CF02", block="B2", mic="M5", device="cuda:0"):
    """Jasper exact match / CER of healthy words decoded from codes alone vs vocoded directly."""
    pipeline, speaker_embs = load_pipeline(checkpoint, device)
    pipeline.use_skip = False                       # codes only, as at inference in the new design
    pipeline.predict_codes = False                  # the gate tests the tokenizer's own codes
    dev = next(pipeline.parameters()).device
    labels = load_word_labels(WORDLIST)
    if reference not in speaker_embs:               # pretrain_best.pt has only LibriTTS speakers
        print(f"No {reference} embedding in this checkpoint; using the mean of all speakers")
        spk = torch.nn.functional.normalize(torch.stack(list(speaker_embs.values())).mean(0), dim=-1)
    else:
        spk = speaker_embs[reference]
    spk = spk.to(dev).reshape(1, -1)
    out = Path(OUT_DIR) / f"gate_{Path(checkpoint).stem}"
    for s in ("direct", "codes"):
        (out / s).mkdir(parents=True, exist_ok=True)

    rows = []
    for p in tqdm(sorted(Path(UA_PATH).rglob(f"{reference}_{block}_*_{mic}.pt")), desc="gate"):
        _, blk, code, _ = _parse_stem(p.stem)
        ref = label_for(labels, blk, code)
        if ref is None:
            continue
        mel = torch.load(p, weights_only=False)["mel"].unsqueeze(0).to(dev)
        T = mel.shape[-1]
        pred = pipeline(mel, torch.zeros(1, T, dtype=torch.bool, device=dev),
                        torch.zeros(1, dtype=torch.long, device=dev), spk, None, False)["predicted"]
        _save_16k(pipeline.mel_to_waveform(0.5 * mel), 22050, out / "direct" / f"{p.stem}.wav")
        _save_16k(pipeline.mel_to_waveform(0.5 * pred), 22050, out / "codes" / f"{p.stem}.wav")
        rows.append({"stem": p.stem, "reference": normalize(ref)})
    df = pd.DataFrame(rows)
    del pipeline
    torch.cuda.empty_cache()

    jasper_model = load_jasper(device)
    for s in ("direct", "codes"):
        df[f"hyp_{s}"] = [normalize(t) for t in
                          transcribe(jasper_model, [str(out / s / f"{st}.wav") for st in df["stem"]])]
    print(f"\n{reference} {block} {mic}: {len(df)} words")
    for s in ("direct", "codes"):
        h = df[f"hyp_{s}"]
        print(f"  {s:7s} exact {100 * (h == df.reference).mean():5.1f}%   empty {100 * (h == '').mean():5.1f}%   "
              f"CER {100 * jiwer.cer(df.reference.tolist(), h.tolist()):5.1f}%")
    return df

## Oracle: true target codes

Decodes each B2 file's stored `target_codes` (CF02's codes in the dysarthric timing) without the recognizer:
the ceiling of the code-prediction design. Pass `original_df` from `run_jasper` to add the original WER on the
same words and UNIT-DSR-style relative WER reduction.

In [ ]:
@torch.no_grad()
def oracle_codes_check(checkpoint, speakers=SPEAKERS, mic="M5", device="cuda:0", original_df=None,
                       merge_classes=None, tokenizer_checkpoint="checkpoints/tokenizer_best.pt"):
    """Jasper on the stored target_codes decoded directly (no recognizer): the ceiling of the design.

    tokenizer_checkpoint: encodes CF02's dtw_target on the fly when a file has no stored target_codes
    from that tokenizer (a fresh runtime has none until precompute_target_codes runs). Must be the
    tokenizer whose codebook the recognizer was trained on.
    merge_classes: int -> each code is replaced by the representative code of its merged class
    (code_merge_map in main.py) before decoding: the ceiling of a recognizer that predicts merged classes.

    checkpoint: a recognizer checkpoint (its frozen codebook must be the one the target_codes came from).
    original_df: optional DataFrame from run_jasper (has 'stem', 'hyp_original').
    """
    pipe, _ = load_pipeline(checkpoint, device)
    dev = next(pipe.parameters()).device
    labels = load_word_labels(WORDLIST)
    merge = code_merge_map(checkpoint, merge_classes) if merge_classes else None
    tag = f"_m{merge_classes}" if merge_classes else ""
    out = Path(OUT_DIR) / f"oracle_{Path(checkpoint).stem}{tag}"
    out.mkdir(parents=True, exist_ok=True)

    tok, tok_tag = None, Path(tokenizer_checkpoint).name
    n_encoded = 0
    rows = []
    for spk in speakers:
        for p in tqdm(sorted(Path(UA_PATH).rglob(f"{spk}_B2_*_{mic}.pt")), desc=spk):
            d = torch.load(p, weights_only=False)
            _, blk, code, _ = _parse_stem(p.stem)
            ref = label_for(labels, blk, code)
            if ref is None or "dtw_target" not in d:
                continue
            if "target_codes" in d and d.get("target_codes_tokenizer") == tok_tag:
                codes = torch.as_tensor(d["target_codes"]).long()                 # (T', G)
            else:
                if tok is None:
                    tok = load_frozen_tokenizer(tokenizer_checkpoint, dev)
                codes = tok.vq(tok.encoder(d["dtw_target"][None].to(dev), None))["codes"][0].cpu()
                n_encoded += 1
            if merge is not None:
                codes = torch.stack([merge["rep"][g][merge["cls"][g][codes[:, g]]] for g in range(codes.shape[1])], 1)
            q = pipe.codes_to_quantized(codes[None].to(dev))
            skip = torch.zeros(1, q.shape[1], pipe.skip_proj.out_features, device=dev)
            cond = pipe.speaker_injector(torch.cat([q, skip], -1), pipe.reference_emb[None])
            mel = pipe.decoder(cond).transpose(1, 2)
            _save_16k(pipe.mel_to_waveform(0.5 * mel), 22050, out / f"{p.stem}.wav")
            rows.append({"stem": p.stem, "speaker": spk, "reference": normalize(ref)})
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No B2 {mic} files with a word label and dtw_target for {speakers} under {UA_PATH}")
    if n_encoded:
        print(f"  encoded {n_encoded} targets on the fly with {tok_tag}")
    del pipe, tok
    torch.cuda.empty_cache()

    jm = load_jasper(device)
    df["hyp_oracle"] = [normalize(t) for t in transcribe(jm, [str(out / f"{s}.wav") for s in df["stem"]])]
    systems = ["oracle"]
    if original_df is not None:
        df = df.merge(original_df[["stem", "hyp_original"]], on="stem", how="left")
        df["hyp_original"] = df["hyp_original"].fillna("")
        systems = ["original", "oracle"]

    table = []
    for key, g in list(df.groupby("speaker")) + [("ALL", df)]:
        row = {"speaker": key, "n_words": len(g)}
        for s in systems:
            refs, hyps = g["reference"].tolist(), g[f"hyp_{s}"].tolist()
            row[f"WER_{s}"] = round(100 * jiwer.wer(refs, hyps), 1)
            row[f"exact_{s}"] = round(100 * (g[f"hyp_{s}"] == g["reference"]).mean(), 1)
            row[f"CER_{s}"] = round(100 * jiwer.cer(refs, hyps), 1)
            row[f"empty_{s}"] = round(100 * (g[f"hyp_{s}"] == "").mean(), 1)
        if original_df is not None:
            o, r = row["WER_original"], row["WER_oracle"]
            row["rel_WER_reduction_%"] = round(100 * (o - r) / o, 1) if o else float("nan")
        table.append(row)
    table = pd.DataFrame(table)
    print(f"\nOracle (true target codes{', merged to ' + str(merge_classes) if merge_classes else ''} "
          f"-> decoder -> HiFi-GAN), B2 {mic}")
    _show(table)
    if original_df is not None:
        spk_rows = table[table["speaker"] != "ALL"]
        print(f"Mean of per-speaker relative WER reductions (UNIT-DSR style): "
              f"{spk_rows['rel_WER_reduction_%'].mean():.1f}%")
    return df, table